<a href="https://colab.research.google.com/github/sophiashiuu/DS340-Final-Project/blob/main/quadrant_mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from google.colab import drive
import pandas as pd

# Loading Data

In [2]:
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/DS 340/Final Project/PMEmo2019/annotations/static_annotations.csv")
lyrics_df = pd.read_csv("/content/drive/MyDrive/DS 340/Final Project/lyrics_clean.csv")
X = np.load("/content/drive/MyDrive/DS 340/Final Project/lyrics_embeddings.npy")

Mounted at /content/drive


In [3]:
print(f"Shape of X: {X.shape}")

Shape of X: (625, 384)


In [4]:
df.head()

,musicId,Arousal(mean),Valence(mean)
0,1,0.4000,0.5750
1,4,0.2625,0.2875
2,5,0.1500,0.2000
3,6,0.5125,0.3500
4,7,0.7000,0.7250


# Adding mood labels

In [5]:
def classify_mood(arousal, valence):
  if arousal < 0.5 and valence < 0.5:
    mood = 0
  elif arousal < 0.5 and valence >= 0.5:
    mood = 1
  elif arousal >= 0.5 and valence < 0.5:
    mood = 2
  else:
    mood = 3

  return mood

In [6]:
df["mood"] = df.apply(lambda row: classify_mood(row["Arousal(mean)"], row["Valence(mean)"]), axis=1)

In [7]:
df.head()

,musicId,Arousal(mean),Valence(mean),mood
0,1,0.4000,0.5750,1
1,4,0.2625,0.2875,0
2,5,0.1500,0.2000,0
3,6,0.5125,0.3500,2
4,7,0.7000,0.7250,3


In [ ]:
df.to_csv("quadrant_data.csv", index=False)

# Restricting datapoints to those that have both spectrogram and lyric data

In [8]:
mask = lyrics_df["musicId"].isin(df["musicId"])

In [9]:
print(mask)

0      True
1      True
2      True
3      True
4      True
       ... 
620    True
621    True
622    True
623    True
624    True
Name: musicId, Length: 625, dtype: bool


In [10]:
X_filtered = []

for i in range(len(mask)):
    if mask.iloc[i] == True:
        X_filtered.append(X[i])

X_filtered = np.array(X_filtered)

In [11]:
lyrics_filtered = lyrics_df[mask].reset_index(drop=True)

In [12]:
print(X_filtered.shape)
print(lyrics_filtered.shape)

(605, 384)
(605, 2)


# Merge with mood data

In [13]:
merged_df = lyrics_filtered.merge(df[["musicId", "mood"]], on="musicId", how="left")

y_filtered = merged_df["mood"].to_numpy()

In [16]:
folder_path = "/content/drive/MyDrive/DS 340/Final Project/"

np.save(folder_path + "X_lyrics_filtered.npy", X_filtered)
np.save(folder_path + "y_lyrics_filtered.npy", y_filtered)
np.save(folder_path + "music_ids_filtered.npy", merged_df["musicId"].to_numpy())